## Read the tissue positions and metadata file
This gives us information on the format of pixel value coordinates.
It also gives us an info on outliers and e.g. how many negative examples entries in the y-position-column there are. 

In [8]:
tissuepositions

,barcode,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres
0,s_008um_00000_00000-1,0,0,0,36591.694839,-242.091470
1,s_008um_00000_00001-1,0,0,1,36591.965228,-212.877209
2,s_008um_00000_00002-1,0,0,2,36592.235617,-183.662943
3,s_008um_00000_00003-1,0,0,3,36592.506006,-154.448672
4,s_008um_00000_00004-1,0,0,4,36592.776395,-125.234396
...,...,...,...,...,...,...
702239,s_008um_00837_00833-1,1,837,833,12357.868203,24323.438541
702240,s_008um_00837_00834-1,1,837,834,12358.136674,24352.657678
702241,s_008um_00837_00835-1,1,837,835,12358.405145,24381.876820
702242,s_008um_00837_00836-1,1,837,836,12358.673616,24411.095967


In [13]:
print("positive pixel values: ", len(tissuepositions[tissuepositions["pxl_col_in_fullres"] > 0]))
print("negative pixel values: ", len(tissuepositions[tissuepositions["pxl_col_in_fullres"] < 0]))
print("in_tissue == 1: ", len(tissuepositions[tissuepositions["in_tissue"] == 1]))
print("in_tissue == 0: ", len(tissuepositions[tissuepositions["in_tissue"] == 0]))

positive pixel values:  698148
negative pixel values:  4096
in_tissue == 1:  507684
in_tissue == 0:  194560


In [20]:
col = "pxl_col_in_fullres"
smallest_gt_zero = tissuepositions.loc[tissuepositions[col] > 0, col].min()
print(f"Smallest positive value in {col}: {smallest_gt_zero}")

smallest_gt_zero = tissuepositions.loc[tissuepositions[col] > 0, col].max()
print(f"Biggest positive value in {col}: {smallest_gt_zero}")

Smallest positive value in pxl_col_in_fullres: 0.01616907989636651
Biggest positive value in pxl_col_in_fullres: 24440.315118149927


## IDK what this does

In [4]:
# Count occurrences of each id
id_count_check = len(results["id"].unique())
id_count = results["id"].value_counts()
print("Unique IDs found in the results:", id_count_check)  # List of unique IDs
print("Counts of each ID:", id_count)  # Counts of each ID

# Filter brands that occur more than once
ids_more_than_once = id_count[id_count > 1]

# If you want just the list of brand names:
id_label = ids_more_than_once.index.tolist()

# If you want the number of such brands:
number_of_ids = len(ids_more_than_once)

print("IDs appearing more than once:")
print(ids_more_than_once)       # brand name and counts
#print(id_label)                 # just brand names
print("Number of Ids: ", number_of_ids)           # count of brands appearing more than once


Unique IDs found in the results: 110016
Counts of each ID: id
ID_85299     7
ID_534315    5
ID_381216    5
ID_84659     5
ID_276597    5
            ..
ID_391996    1
ID_390550    1
ID_391503    1
ID_390132    1
ID_388651    1
Name: count, Length: 110016, dtype: int64
IDs appearing more than once:
id
ID_85299     7
ID_534315    5
ID_381216    5
ID_84659     5
ID_276597    5
            ..
ID_54843     2
ID_535021    2
ID_531992    2
ID_401223    2
ID_540878    2
Name: count, Length: 3687, dtype: int64
Number of Ids:  3687


In [1]:
import pandas as pd
df1 = pd.read_csv("uni/notebooks/metadata/WSI_patch_metadata1.csv")
df2 = pd.read_csv("uni/notebooks/metadata/WSI_patch_metadata2.csv")

merged_df = pd.concat([df1, df2], axis=0, ignore_index=True)

merged_df.to_csv("uni/notebooks/metadata/WSI_patch_metadata.csv", index=False)

In [2]:
import scanpy as sc
import anndata as ad

# Load both .h5ad files
adata1 = sc.read_h5ad("uni/notebooks/metadata/WSI_patch_embeddings1.h5ad")
adata2 = sc.read_h5ad("uni/notebooks/metadata/WSI_patch_embeddings2.h5ad")

# Concatenate along observations (rows)
adata_merged = ad.concat([adata1, adata2], axis=0, join="outer")  
# join="outer" keeps all obs/var, join="inner" keeps only common

# Save
adata_merged.write_h5ad("uni/notebooks/metadata/WSI_patch_embeddings.h5ad")


## Checking coordinates of filtered patches and Nuclei

In [1]:
import os
import scanpy as sc
import pandas as pd
import geopandas as gpd


person = "CRC_P1"
bin_size = "002um"

uni_folder = "uni/notebooks/metadata"
uni_patch_embeddings_filename = "filtered_WSI_patch_embeddings.h5ad"

cell_seg_folder = f"Cell_Segmentation/output/{person}/{bin_size}/custom_binning"
filtered_bins_filename = "filtered_custom_bins.h5ad"
nuclei_genes_filtered_filename = "count_area_filtered_adata.h5ad"
nuclei_polys_filename = "nuclei_geometry.parquet"

In [2]:
# Loading the data
adata_patches = sc.read_h5ad(os.path.join(uni_folder, uni_patch_embeddings_filename))
#adata_bins = sc.read_h5ad(os.path.join(cell_seg_folder, filtered_bins_filename))
nuclei_positions = gpd.read_parquet(os.path.join(cell_seg_folder, nuclei_polys_filename))
adata_filtered_nuclei = sc.read_h5ad(os.path.join(cell_seg_folder, nuclei_genes_filtered_filename))

# merge nuclei positions with adata_filtered_nuclei
adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on="id", how="left")


/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [ ]:
print("Min x:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "X"].min())
print("Max x:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "X"].max())
print("Min y:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "Y"].min())
print("Max y:", adata_patches.obs.loc[adata_patches.obs["Slide_ID"]==person, "Y"].max())

Min x: 448
Max x: 24192


In [10]:
# filter rows/columns > 0
row_vals = tissuepositions.loc[tissuepositions['pxl_row_in_fullres'] > 0, 'pxl_row_in_fullres']
col_vals = tissuepositions.loc[tissuepositions['pxl_col_in_fullres'] > 0, 'pxl_col_in_fullres']

# compute min and max
min_row, max_row = row_vals.min(), row_vals.max()
min_col, max_col = col_vals.min(), col_vals.max()

print(f"Row -> min > 0: {min_row}, max: {max_row}")
print(f"Column -> min > 0: {min_col}, max: {max_col}")

Row -> min > 0: 12137.797400349906, max: 36828.94892524755
Column -> min > 0: 0.0003797818750692132, max: 24436.630813568463


In [8]:
import scanpy as sc
import pandas as pd
import os

h5ad_test = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/filtered_custom_bins.h5ad")

In [7]:
import scanpy as sc
import pandas as pd
import geopandas as gpd

person = "CRC_P1"
bin_size = "002um"

adata_filtered_nuclei = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/002um/count_area_filtered_adata.h5ad")
nuclei_positions = gpd.read_parquet("Cell_Segmentation/output/merged_custom_binning/002um/nuclei_geometry.parquet")

#adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on=["id", "Slide_ID"], how="left")
adata_filtered_nuclei.obs = adata_filtered_nuclei.obs.merge(nuclei_positions, on=["Slide_ID", "id"], how="left")
adata_filtered_nuclei = adata_filtered_nuclei[adata_filtered_nuclei.obs["Slide_ID"] == person]


# gives us the overview over each nucleus for each person, with its geometry
# adata_filtered_nuclei.obs

In [ ]:
nuclei_positions = nuclei_positions[nuclei_positions["Slide_ID"] == person]
# nuclei_positions

# Get the bounding box of the whole dataset
minx, miny, maxx, maxy = nuclei_positions.total_bounds

print("Bounds from Nuclei Positiosn Parquet file:")
print("Min X:", minx)
print("Min Y:", miny)
print("Max X:", maxx)
print("Max Y:", maxy)


minx2, miny2, maxx2, maxy2 = adata_filtered_nuclei.total_bounds

print("Bounds from merged adata file:")
print("Min X:", minx2)
print("Min Y:", miny2)
print("Max X:", maxx2)
print("Max Y:", maxy2)


Bounds from Nuclei Positiosn Parquet file:
Min X: 680.6541748046875
Min Y: 14102.367919921875
Max X: 24408.420166015625
Max Y: 36828.15768432617
Bounds from merged adata file:
Min X: 680.6541748046875
Min Y: 14102.367919921875
Max X: 24408.420166015625
Max Y: 36828.15768432617


# Check Anndata structure for Uni and StarDist

In [1]:
import scanpy as sc
import geopandas as gpd

In [ ]:
unidata_multi = sc.read_h5ad("uni/notebooks/metadata/filtered_WSI_patch_embeddings.h5ad")
unidata_multi.obs

,Slide_ID,Patch_ID,X,Y,leiden_0.2,leiden_0.5,leiden_1.0,leiden_2.0
CRC_P1_patch_295.png,CRC_P1,patch_295.png,448,28713,6,5,5,8
CRC_P1_patch_296.png,CRC_P1,patch_296.png,448,28937,6,5,5,25
CRC_P1_patch_297.png,CRC_P1,patch_297.png,448,29161,6,5,5,25
CRC_P1_patch_298.png,CRC_P1,patch_298.png,448,29385,6,5,5,25
CRC_P1_patch_299.png,CRC_P1,patch_299.png,448,29609,6,5,5,25
...,...,...,...,...,...,...,...,...
Normal_P5_patch_11877.png,Normal_P5,patch_11877.png,65876,49773,1,1,2,2
Normal_P5_patch_11878.png,Normal_P5,patch_11878.png,65876,49997,1,1,2,2
Normal_P5_patch_11879.png,Normal_P5,patch_11879.png,65876,50221,1,1,2,2
Normal_P5_patch_11880.png,Normal_P5,patch_11880.png,65876,50445,1,1,2,2


In [2]:
person = "CRC_P1"
single_countareafilteredadata = sc.read_h5ad("Cell_Segmentation/output/CRC_P1/002um/custom_binning/count_area_filtered_adata.h5ad")
single_countareafilteredadata.obs

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


,id,Slide_ID,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,clusters
5,ID_52421,CRC_P1,97,4.584967,112.0,4.727388,58.035714,100.000000,100.000000,100.0,1
13,ID_52429,CRC_P1,106,4.672829,121.0,4.804021,53.719008,95.041322,100.000000,100.0,1
23,ID_52439,CRC_P1,136,4.919981,149.0,5.010635,42.281879,75.838926,100.000000,100.0,1
31,ID_52447,CRC_P1,88,4.488636,137.0,4.927254,72.262774,100.000000,100.000000,100.0,1
35,ID_52451,CRC_P1,95,4.564348,111.0,4.718499,59.459459,100.000000,100.000000,100.0,1
...,...,...,...,...,...,...,...,...,...,...,...
289375,ID_735550,CRC_P1,119,4.787492,127.0,4.852030,45.669291,85.039370,100.000000,100.0,0
289378,ID_735647,CRC_P1,94,4.553877,105.0,4.663439,58.095238,100.000000,100.000000,100.0,0
289383,ID_735866,CRC_P1,111,4.718499,142.0,4.962845,57.042254,92.253521,100.000000,100.0,2
289391,ID_736660,CRC_P1,85,4.454347,120.0,4.795791,70.833333,100.000000,100.000000,100.0,2


In [4]:
multi_countareafilteredadata = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/002um/count_area_filtered_adata.h5ad")
multi_countareafilteredadata.obs

,id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,clusters,Slide_ID
CRC_P1_5,ID_52421,97,4.584967,112.0,4.727388,58.035714,100.000000,100.000000,100.000000,1,CRC_P1
CRC_P1_13,ID_52429,106,4.672829,121.0,4.804021,53.719008,95.041322,100.000000,100.000000,1,CRC_P1
CRC_P1_23,ID_52439,136,4.919981,149.0,5.010635,42.281879,75.838926,100.000000,100.000000,1,CRC_P1
CRC_P1_31,ID_52447,88,4.488636,137.0,4.927254,72.262774,100.000000,100.000000,100.000000,1,CRC_P1
CRC_P1_35,ID_52451,95,4.564348,111.0,4.718499,59.459459,100.000000,100.000000,100.000000,1,CRC_P1
...,...,...,...,...,...,...,...,...,...,...,...
Normal_P5_222609,ID_666491,117,4.770685,138.0,4.934474,51.449275,87.681159,100.000000,100.000000,1,Normal_P5
Normal_P5_222610,ID_666492,144,4.976734,370.0,5.916202,74.594595,88.108108,100.000000,100.000000,0,Normal_P5
Normal_P5_222613,ID_666496,561,6.331502,726.0,6.588926,25.482094,36.501377,50.275482,91.597796,0,Normal_P5
Normal_P5_222614,ID_666497,158,5.068904,182.0,5.209486,40.659341,68.131868,100.000000,100.000000,0,Normal_P5


In [3]:
single_filteredcustombins = sc.read_h5ad("Cell_Segmentation/output/CRC_P1/002um/custom_binning/filtered_custom_bins.h5ad")
single_filteredcustombins.obs

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/anndata.py:1820: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,index_x,index_y,id,is_within_polygon,is_not_in_an_polygon_overlap,Slide_ID
s_002um_01867_02041-1,1,1867,2041,23101.556076,14781.241485,s_002um_01867_02041-1,s_002um_01867_02041-1,ID_239410,True,True,CRC_P1
s_002um_02109_01204-1,1,2109,1204,21277.315954,8684.120489,s_002um_02109_01204-1,s_002um_02109_01204-1,ID_146203,True,True,CRC_P1
s_002um_02496_03085-1,1,2496,3085,18576.619085,22449.992628,s_002um_02496_03085-1,s_002um_02496_03085-1,ID_172530,True,True,CRC_P1
s_002um_00481_01245-1,1,481,1245,33172.972720,8872.760237,s_002um_00481_01245-1,s_002um_00481_01245-1,ID_524803,True,True,CRC_P1
s_002um_00727_01427-1,1,727,1427,31388.196866,10218.836278,s_002um_00727_01427-1,s_002um_00727_01427-1,ID_383635,True,True,CRC_P1
...,...,...,...,...,...,...,...,...,...,...,...
s_002um_01041_01763-1,1,1041,1763,29117.030951,12694.396181,s_002um_01041_01763-1,s_002um_01041_01763-1,ID_303658,True,True,CRC_P1
s_002um_01051_01858-1,1,1051,1858,29050.385985,13388.973597,s_002um_01051_01858-1,s_002um_01051_01858-1,ID_299704,True,True,CRC_P1
s_002um_02298_02981-1,1,2298,2981,20016.140671,21676.806463,s_002um_02298_02981-1,s_002um_02298_02981-1,ID_171887,True,True,CRC_P1
s_002um_01377_01056-1,1,1377,1056,26614.803940,7553.299400,s_002um_01377_01056-1,s_002um_01377_01056-1,ID_290533,True,True,CRC_P1


In [6]:
multi_filteredcustombins = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/002um/filtered_custom_bins.h5ad")
multi_filteredcustombins.obs

,in_tissue,array_row,array_col,pxl_row_in_fullres,pxl_col_in_fullres,index_x,index_y,id,is_within_polygon,is_not_in_an_polygon_overlap,Slide_ID
CRC_P1_s_002um_01867_02041-1,1,1867,2041,23101.556076,14781.241485,s_002um_01867_02041-1,s_002um_01867_02041-1,ID_239410,True,True,CRC_P1
CRC_P1_s_002um_02109_01204-1,1,2109,1204,21277.315954,8684.120489,s_002um_02109_01204-1,s_002um_02109_01204-1,ID_146203,True,True,CRC_P1
CRC_P1_s_002um_02496_03085-1,1,2496,3085,18576.619085,22449.992628,s_002um_02496_03085-1,s_002um_02496_03085-1,ID_172530,True,True,CRC_P1
CRC_P1_s_002um_00481_01245-1,1,481,1245,33172.972720,8872.760237,s_002um_00481_01245-1,s_002um_00481_01245-1,ID_524803,True,True,CRC_P1
CRC_P1_s_002um_00727_01427-1,1,727,1427,31388.196866,10218.836278,s_002um_00727_01427-1,s_002um_00727_01427-1,ID_383635,True,True,CRC_P1
...,...,...,...,...,...,...,...,...,...,...,...
Normal_P5_s_002um_00804_03277-1,1,804,3277,32351.711109,42237.213333,s_002um_00804_03277-1,s_002um_00804_03277-1,ID_396224,True,True,Normal_P5
Normal_P5_s_002um_01533_01973-1,1,1533,1973,37690.479472,51755.407138,s_002um_01533_01973-1,s_002um_01533_01973-1,ID_498911,True,True,Normal_P5
Normal_P5_s_002um_00296_02357-1,1,296,2357,28650.366640,48963.184135,s_002um_00296_02357-1,s_002um_00296_02357-1,ID_315195,True,True,Normal_P5
Normal_P5_s_002um_01141_01847-1,1,1141,1847,34828.311124,52679.930388,s_002um_01141_01847-1,s_002um_01141_01847-1,ID_416686,True,True,Normal_P5


In [4]:
single_polys = gpd.read_parquet("Cell_Segmentation/output/CRC_P1/002um/custom_binning/nuclei_geometry.parquet")
single_polys

,id,geometry,Slide_ID
5,ID_52421,"POLYGON ((14490.007 14852.000, 14490.093 14853...",CRC_P1
13,ID_52429,"POLYGON ((14409.892 14808.000, 14408.815 14810...",CRC_P1
23,ID_52439,"POLYGON ((14437.344 14816.000, 14434.664 14818...",CRC_P1
31,ID_52447,"POLYGON ((14055.935 14792.000, 14055.610 14793...",CRC_P1
35,ID_52451,"POLYGON ((14464.044 14820.000, 14464.270 14822...",CRC_P1
...,...,...,...
289375,ID_735550,"POLYGON ((23650.012 36770.000, 23649.842 36772...",CRC_P1
289378,ID_735647,"POLYGON ((23570.196 36782.000, 23568.974 36783...",CRC_P1
289383,ID_735866,"POLYGON ((21905.758 36780.000, 21906.443 36782...",CRC_P1
289391,ID_736660,"POLYGON ((23536.900 36800.000, 23537.887 36801...",CRC_P1


In [6]:
multi_polys = gpd.read_parquet("Cell_Segmentation/output/merged_custom_binning/002um/nuclei_geometry.parquet")
multi_polys

,id,geometry,Slide_ID
0,ID_52421,"POLYGON ((14490.007 14852.000, 14490.093 14853...",CRC_P1
1,ID_52429,"POLYGON ((14409.892 14808.000, 14408.815 14810...",CRC_P1
2,ID_52439,"POLYGON ((14437.344 14816.000, 14434.664 14818...",CRC_P1
3,ID_52447,"POLYGON ((14055.935 14792.000, 14055.610 14793...",CRC_P1
4,ID_52451,"POLYGON ((14464.044 14820.000, 14464.270 14822...",CRC_P1
...,...,...,...
456876,ID_666491,"POLYGON ((52314.505 45776.000, 52314.601 45776...",Normal_P5
456877,ID_666492,"POLYGON ((51664.354 46284.000, 51664.362 46284...",Normal_P5
456878,ID_666496,"POLYGON ((52024.459 46084.000, 52024.903 46084...",Normal_P5
456879,ID_666497,"POLYGON ((51702.852 45716.000, 51702.877 45716...",Normal_P5


In [9]:
assigned_nuclei = sc.read_h5ad("Cell_Segmentation/output/merged_custom_binning/002um/patch_assigned_count_area_filtered_adata.h5ad")
assigned_nuclei.obs

/home/charlottewolf/miniconda3/envs/NucleiSeg/lib/python3.11/site-packages/anndata/_core/aligned_df.py:67: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


,id,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,clusters,Slide_ID,Patch_ID
0,ID_52421,97,4.584967,112.0,4.727388,58.035714,100.000000,100.000000,100.000000,1,CRC_P1,patch_7053.png
1,ID_52429,106,4.672829,121.0,4.804021,53.719008,95.041322,100.000000,100.000000,1,CRC_P1,patch_7052.png
2,ID_52439,136,4.919981,149.0,5.010635,42.281879,75.838926,100.000000,100.000000,1,CRC_P1,patch_7052.png
3,ID_52447,88,4.488636,137.0,4.927254,72.262774,100.000000,100.000000,100.000000,1,CRC_P1,patch_6832.png
4,ID_52451,95,4.564348,111.0,4.718499,59.459459,100.000000,100.000000,100.000000,1,CRC_P1,patch_7052.png
...,...,...,...,...,...,...,...,...,...,...,...,...
456876,ID_666491,117,4.770685,138.0,4.934474,51.449275,87.681159,100.000000,100.000000,1,Normal_P5,patch_5210.png
456877,ID_666492,144,4.976734,370.0,5.916202,74.594595,88.108108,100.000000,100.000000,0,Normal_P5,patch_4885.png
456878,ID_666496,561,6.331502,726.0,6.588926,25.482094,36.501377,50.275482,91.597796,0,Normal_P5,patch_5102.png
456879,ID_666497,158,5.068904,182.0,5.209486,40.659341,68.131868,100.000000,100.000000,0,Normal_P5,patch_4882.png


# Add extra column with slide name

In [ ]:
import scanpy as sc 
import pandas as pd

check = sc.read_h5ad("Cell_Segmentation/output/CRC_P5/002um/custom_binning/nuclei_grouped_UMI_count.h5ad")
# check.obs
test = pd.read_parquet("Cell_Segmentation/output/CRC_P5/002um/custom_binning/nuclei_geometry.parquet")
test

In [ ]:
import scanpy as sc 
import pandas as pd


people = ["CRC_P1", "CRC_P2", "CRC_P5", "Normal_P3", "Normal_P5"]
file_names = ["nuclei_geometry.parquet", "filtered_custom_bins.h5ad", "nuclei_grouped_UMI_count.h5ad", "count_area_filtered_adata.h5ad"]

for person in people:
    for file_name in file_names:
        if file_name.endswith(".h5ad"):
            current_file = sc.read_h5ad(f"Cell_Segmentation/output/{person}/002um/custom_binning/{file_name}")
            print("Currently reading file: ", current_file)
            
            if "Slide_ID" not in current_file.obs.columns:
                current_file.obs["Slide_ID"] = person
                current_file.write(f"Cell_Segmentation/output/{person}/002um/custom_binning/{file_name}")

        elif file_name.endswith(".parquet"):
            current_file = pd.read_parquet(f"Cell_Segmentation/output/{person}/002um/custom_binning/{file_name}")
            print("Currently reading file: ", current_file)
            
            if "Slide_ID" not in current_file.columns:
                current_file["Slide_ID"] = person
                current_file.to_parquet(f"Cell_Segmentation/output/{person}/002um/custom_binning/{file_name}")

        else:
            print("WTF")
